In [2]:
pip install -q -U "langchain[google-genai]"

Loading the LLM: Choosing the right language model

In [4]:
import os
from langchain.chat_models import init_chat_model
from google.colab import userdata



os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')


model = init_chat_model("google_genai:gemini-2.5-flash-lite")

In [5]:
response = model.invoke("Why do parrots talk?")

In [8]:
response.content

'Parrots talk for a variety of fascinating reasons, and it\'s not just about mimicking human sounds. It\'s a complex behavior rooted in their natural instincts and social lives. Here\'s a breakdown of the main reasons:\n\n**1. Social Bonding and Communication:**\n\n*   **Flock Behavior:** Parrots are highly social animals that live in large flocks in the wild. Within these flocks, they use a variety of vocalizations – squawks, chirps, clicks, and whistles – to communicate with each other. This communication is crucial for:\n    *   **Maintaining Flock Cohesion:** Keeping track of each other, especially when foraging or flying.\n    *   **Warning of Predators:** Alerting the group to danger.\n    *   **Establishing Social Hierarchy:** Communicating dominance and submission.\n    *   **Finding Mates:** Attracting and interacting with potential partners.\n*   **Mimicry as a Social Tool:** In their natural environment, parrots are excellent mimics of sounds they hear around them, including

**Function**

In [9]:
def add_numbers(inputs:str) -> dict:
    """
    Adds a list of numbers provided in the input dictionary or extracts numbers from a string.

    Parameters:
    - inputs (str):
    string, it should contain numbers that can be extracted and summed.

    Returns:
    - dict: A dictionary with a single key "result" containing the sum of the numbers.

    Example Input (Dictionary):
    {"numbers": [10, 20, 30]}

    Example Input (String):
    "Add the numbers 10, 20, and 30."

    Example Output:
    {"result": 60}
    """
    numbers = [int(x) for x in inputs.replace(",", "").split() if x.isdigit()]


    result = sum(numbers)
    return {"result": result}

In [11]:
add_numbers("1 and two 2")

{'result': 3}


## Tool
The `Tool` class in LangChain serves as a structured wrapper that converts regular Python functions into agent-compatible tools. Each tool needs three key components:
1. A name that identifies the tool
2. The function that performs the actual operation
3. A description that helps the agent understand when to use the tool

Testing section improvement:




In [12]:
from langchain.tools import tool

@tool
def add_numbers(inputs:str) -> dict:
    """
    Adds a list of numbers provided in the input dictionary or extracts numbers from a string.

    Parameters:
    - inputs (str):
    string, it should contain numbers that can be extracted and summed.

    Returns:
    - dict: A dictionary with a single key "result" containing the sum of the numbers.

    Example Input (Dictionary):
    {"numbers": [10, 20, 30]}

    Example Input (String):
    "Add the numbers 10, 20, and 30."

    Example Output:
    {"result": 60}
    """
    numbers = [int(x) for x in inputs.replace(",", "").split() if x.isdigit()]


    result = sum(numbers)
    return {"result": result}

print("tool object",add_numbers)

tool object name='add_numbers' description='Adds a list of numbers provided in the input dictionary or extracts numbers from a string.\n\nParameters:\n- inputs (str): \nstring, it should contain numbers that can be extracted and summed.\n\nReturns:\n- dict: A dictionary with a single key "result" containing the sum of the numbers.\n\nExample Input (Dictionary):\n{"numbers": [10, 20, 30]}\n\nExample Input (String):\n"Add the numbers 10, 20, and 30."\n\nExample Output:\n{"result": 60}' args_schema=<class 'langchain_core.utils.pydantic.add_numbers'> func=<function add_numbers at 0x7d7a089fa2a0>


In [14]:
# Tool name
print("Tool Name:")
print(add_numbers.name)

# Tool description
print("Tool Description:")
print(add_numbers.description)

# Tool function
print("Tool Function:")
print(add_numbers.invoke)


Tool Name:
add_numbers
Tool Description:
Adds a list of numbers provided in the input dictionary or extracts numbers from a string.

Parameters:
- inputs (str): 
string, it should contain numbers that can be extracted and summed.

Returns:
- dict: A dictionary with a single key "result" containing the sum of the numbers.

Example Input (Dictionary):
{"numbers": [10, 20, 30]}

Example Input (String):
"Add the numbers 10, 20, and 30."

Example Output:
{"result": 60}
Tool Function:
<bound method BaseTool.invoke of StructuredTool(name='add_numbers', description='Adds a list of numbers provided in the input dictionary or extracts numbers from a string.\n\nParameters:\n- inputs (str): \nstring, it should contain numbers that can be extracted and summed.\n\nReturns:\n- dict: A dictionary with a single key "result" containing the sum of the numbers.\n\nExample Input (Dictionary):\n{"numbers": [10, 20, 30]}\n\nExample Input (String):\n"Add the numbers 10, 20, and 30."\n\nExample Output:\n{"resu

Calling the tool

In [16]:
print("Calling Tool Function:")
test_input = "10 hello 20 30 a b"
print(add_numbers.invoke(test_input))

Calling Tool Function:
{'result': 60}


In [18]:



print(f"Has Schema: {hasattr(add_numbers, 'args_schema')}")
print(f"Args Schema Info: {add_numbers.args}")

Has Schema: True
Args Schema Info: {'inputs': {'title': 'Inputs', 'type': 'string'}}


In this example, the tool has two inputs: a string containing the numbers to add, and a second boolean input that determines whether to sum the absolute values of those numbers.


In [19]:
from typing import List

@tool
def add_numbers_with_options(numbers: List[float], absolute: bool = False) -> float:
    """
    Adds a list of numbers provided as input.

    Parameters:
    - numbers (List[float]): A list of numbers to be summed.
    - absolute (bool): If True, use the absolute values of the numbers before summing.

    Returns:
    - float: The total sum of the numbers.
    """
    if absolute:
        numbers = [abs(n) for n in numbers]
    return sum(numbers)

In [20]:
print(f"Args Schema Info: {add_numbers_with_options.args}")
print(f"Args Schema Info: {add_numbers.args}")

Args Schema Info: {'numbers': {'items': {'type': 'number'}, 'title': 'Numbers', 'type': 'array'}, 'absolute': {'default': False, 'title': 'Absolute', 'type': 'boolean'}}
Args Schema Info: {'inputs': {'title': 'Inputs', 'type': 'string'}}


In [21]:
print(add_numbers_with_options.invoke({"numbers":[-1.1,-2.1,-3.0],"absolute":False}))
print(add_numbers_with_options.invoke({"numbers":[-1.1,-2.1,-3.0],"absolute":True}))

-6.2
6.2


## Improved tool return types with Python typing

When creating tools, you must accurately specify their return values. This helps the agent understand and handle different possible outputs.



In [22]:
from typing import Dict, Union

@tool
def sum_numbers_with_complex_output(inputs: str) -> Dict[str, Union[float, str]]:
    """
    Extracts and sums all integers and decimal numbers from the input string.

    Parameters:
    - inputs (str): A string that may contain numeric values.

    Returns:
    - dict: A dictionary with the key "result". If numbers are found, the value is their sum (float).
            If no numbers are found or an error occurs, the value is a corresponding message (str).

    Example Input:
    "Add 10, 20.5, and -3."

    Example Output:
    {"result": 27.5}
    """
    matches = re.findall(r'-?\d+(?:\.\d+)?', inputs)
    if not matches:
        return {"result": "No numbers found in input."}
    try:
        numbers = [float(num) for num in matches]
        total = sum(numbers)
        return {"result": total}
    except Exception as e:
        return {"result": f"Error during summation: {str(e)}"}

In [23]:
@tool
def sum_numbers_from_text(inputs: str) -> float:
    """
    Adds a list of numbers provided in the input string.

    Args:
        text: A string containing numbers that should be extracted and summed.

    Returns:
        The sum of all numbers found in the input.
    """
    # Use regular expressions to extract all numbers from the input
    numbers = [int(num) for num in re.findall(r'\d+', inputs)]
    result = sum(numbers)
    return result

### `initialize_agent`

When you set up an agent, you're connecting tools and an LLM to work together seamlessly. The agent uses the LLM to understand what needs to be done and decides which tool to use based on the task. Here's a simple overview of the key parts:


#### **Relationship between agent and LLM**
- The **agent** acts as the decision-maker, figuring out which tools to use and when.
- The **LLM** is the reasoning engine. It:
  - Interprets the user's input.
  - Helps the agent make decisions.
  - Generates a response based on the output of the tools.

Think of the agent as the manager assigning tasks and the LLM as the brain solving problems or delegating work.

---

#### **Key parameters of `initialize_agent`**

1. **`tools`**- see above

2.  **`llm`** see above

3. **`agent`**:
   - Specifies the reasoning framework for the agent.
   - `"zero-shot-react-description"` enables:
     - **Zero-shot reasoning**: The agent can solve tasks it hasn't seen before by thinking through the problem step by step.
     - **React framework**: A logical loop of:
       - **Reason** → Think about the task.
       - **Act** → Use a tool to perform an action.
       - **Observe** → Check the tool's output.
       - **Plan** → Decide what to do next.

4. **`verbose`**:
   - If `True`, it prints detailed logs of the agent’s thought process.
   - Useful for debugging or understanding how the agent makes decisions.




In [24]:
from langchain.agents import create_agent

agent = create_agent(model, tools=[add_numbers])


In [35]:
response = agent.invoke({
    "messages": [
        {"role": "user", "content": "In 2023, the US GDP was approximately $27.72 trillion, while Canada's was around $2.14 trillion and Mexico's was about $1.79 trillion. What is the total?"}
    ]
})

In [33]:
print(response)

{'messages': [HumanMessage(content="In 2023, the US GDP was approximately $27.72 trillion, while Canada's was around $2.14 trillion and Mexico's was about $1.79 trillion. What is the total?", additional_kwargs={}, response_metadata={}, id='793cfaa4-b4c3-4cee-a43d-8629642813ab'), AIMessage(content='', additional_kwargs={'function_call': {'name': 'add_numbers', 'arguments': '{"inputs": "27.72 trillion, 2.14 trillion, 1.79 trillion"}'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019e50bd-7339-7280-8c6d-8346fb02d534-0', tool_calls=[{'name': 'add_numbers', 'args': {'inputs': '27.72 trillion, 2.14 trillion, 1.79 trillion'}, 'id': '3748076d-94ed-46c1-847f-63edeb480821', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 216, 'output_tokens': 34, 'total_tokens': 250, 'input_token_details': {'cache_read': 0}}), ToolMessage(content='{"result": 0}', name='add_nu

Structured chat zero shot react-description

In [38]:
import re

agent = create_agent(
    model=model,
    tools=[sum_numbers_from_text]
)

response = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Add 10, 20 and 30"
        }
    ]
})

print(response)

{'messages': [HumanMessage(content='Add 10, 20 and 30', additional_kwargs={}, response_metadata={}, id='c7d09c18-1182-4595-835c-f5f1ad5996c6'), AIMessage(content='', additional_kwargs={'function_call': {'name': 'sum_numbers_from_text', 'arguments': '{"inputs": "10, 20, 30"}'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019e50c3-f7ed-7dd2-a32f-ea2feeb906c7-0', tool_calls=[{'name': 'sum_numbers_from_text', 'args': {'inputs': '10, 20, 30'}, 'id': 'a418ce5d-c06f-4b13-98fa-adbb18a25eef', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 95, 'output_tokens': 28, 'total_tokens': 123, 'input_token_details': {'cache_read': 0}}), ToolMessage(content='60', name='sum_numbers_from_text', id='d0519115-2d1f-43a1-8f05-432778537396', tool_call_id='a418ce5d-c06f-4b13-98fa-adbb18a25eef'), AIMessage(content='60', additional_kwargs={}, response_metadata={'finish_reason':

In [40]:

complex_agent = create_agent(
    model=model,
    tools=[sum_numbers_with_complex_output]
)

response = complex_agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Add 10, 20 and 30"
        }
    ]
})

print(response)

{'messages': [HumanMessage(content='Add 10, 20 and 30', additional_kwargs={}, response_metadata={}, id='424b8127-991b-4430-bea0-043fbf756563'), AIMessage(content='', additional_kwargs={'function_call': {'name': 'sum_numbers_with_complex_output', 'arguments': '{"inputs": "Add 10, 20 and 30"}'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019e50c6-6e2b-76c0-8d2c-7caac9d110ed-0', tool_calls=[{'name': 'sum_numbers_with_complex_output', 'args': {'inputs': 'Add 10, 20 and 30'}, 'id': 'ef935536-0a35-4716-a053-756355e7df18', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 175, 'output_tokens': 32, 'total_tokens': 207, 'input_token_details': {'cache_read': 0}}), ToolMessage(content='{"result": 60.0}', name='sum_numbers_with_complex_output', id='0adf05ce-2bfa-4eae-b369-a42ec3b2e03b', tool_call_id='ef935536-0a35-4716-a053-756355e7df18'), AIMessage(content='I c

## Orchestrating multiple tools with an agent: Mathematical toolkit
In real-world applications, a single tool is often insufficient to address the complexity and diversity of user requests. Tasks such as data analysis, performing calculations, or retrieving specific information require specialized capabilities that cannot be fulfilled by a single function. By equipping an agent with multiple tools, each tailored to a distinct purpose, you'll create a system that can dynamically select and utilize the appropriate tool based on the user’s query. This approach enhances the flexibility and scalability of the AI, allowing it to handle a broad spectrum of tasks with precision and efficiency. The orchestration of multiple tools ensures that the agent can seamlessly manage complex workflows, making it an essential framework for building robust and versatile AI systems.

To demonstrate this concept, let’s create additional tools, i.e, a mathematical toolkit. In addition to the addition tool, you will now create tools for subtraction, multiplication, and division. These tools will be integrated into an agent capable of handling various mathematical queries, showcasing how multiple tools can work together within a single AI system.

### Subtraction tool
The subtraction tool is designed to take a list of numbers and return the result of subtracting all subsequent numbers from the first number. This tool is particularly useful for handling queries involving differences, such as "What is 100 minus 20 and then minus 10?".


In [41]:
@tool
def subtract_numbers(inputs: str) -> dict:
    """
    Extracts numbers from a string, negates the first number, and successively subtracts
    the remaining numbers in the list.

    This function is designed to handle input in string format, where numbers are separated
    by spaces, commas, or other delimiters. It parses the string, extracts valid numeric values,
    and performs a step-by-step subtraction operation starting with the first number negated.

    Parameters:
    - inputs (str):
      A string containing numbers to subtract. The string may include spaces, commas, or
      other delimiters between the numbers.

    Returns:
    - dict:
      A dictionary containing the key "result" with the calculated difference as its value.
      If no valid numbers are found in the input string, the result defaults to 0.

    Example Input:
    "100, 20, 10"

    Example Output:
    {"result": -130}

    Notes:
    - Non-numeric characters in the input are ignored.
    - If the input string contains only one valid number, the result will be that number negated.
    - Handles a variety of delimiters (e.g., spaces, commas) but does not validate input formats
      beyond extracting numeric values.
    """
    # Extract numbers from the string
    numbers = [int(num) for num in inputs.replace(",", "").split() if num.isdigit()]

    # If no numbers are found, return 0
    if not numbers:
        return {"result": 0}

    # Start with the first number negated
    result = -1 * numbers[0]

    # Subtract all subsequent numbers
    for num in numbers[1:]:
        result -= num

    return {"result": result}

In [42]:
print("Name: \n", subtract_numbers.name)
print("Description: \n", subtract_numbers.description)
print("Args: \n", subtract_numbers.args)

Name: 
 subtract_numbers
Description: 
 Extracts numbers from a string, negates the first number, and successively subtracts 
the remaining numbers in the list.

This function is designed to handle input in string format, where numbers are separated 
by spaces, commas, or other delimiters. It parses the string, extracts valid numeric values, 
and performs a step-by-step subtraction operation starting with the first number negated.

Parameters:
- inputs (str): 
  A string containing numbers to subtract. The string may include spaces, commas, or 
  other delimiters between the numbers.

Returns:
- dict: 
  A dictionary containing the key "result" with the calculated difference as its value. 
  If no valid numbers are found in the input string, the result defaults to 0.

Example Input:
"100, 20, 10"

Example Output:
{"result": -130}

Notes:
- Non-numeric characters in the input are ignored.
- If the input string contains only one valid number, the result will be that number negated.
- Han

In [43]:
print("Calling Tool Function:")
test_input = "10 20 30 and four a b"
print(subtract_numbers.invoke(test_input))  # Example

Calling Tool Function:
{'result': -60}


In [44]:
# Multiplication Tool
@tool
def multiply_numbers(inputs: str) -> dict:
    """
    Extracts numbers from a string and calculates their product.

    Parameters:
    - inputs (str): A string containing numbers separated by spaces, commas, or other delimiters.

    Returns:
    - dict: A dictionary with the key "result" containing the product of the numbers.

    Example Input:
    "2, 3, 4"

    Example Output:
    {"result": 24}

    Notes:
    - If no numbers are found, the result defaults to 1 (neutral element for multiplication).
    """
    # Extract numbers from the string
    numbers = [int(num) for num in inputs.replace(",", "").split() if num.isdigit()]
    print(numbers)

    # If no numbers are found, return 1
    if not numbers:
        return {"result": 1}

    # Calculate the product of the numbers
    result = 1
    for num in numbers:
        result *= num
        print(num)

    return {"result": result}

In [45]:
# Division Tool
@tool
def divide_numbers(inputs: str) -> dict:
    """
    Extracts numbers from a string and calculates the result of dividing the first number
    by the subsequent numbers in sequence.

    Parameters:
    - inputs (str): A string containing numbers separated by spaces, commas, or other delimiters.

    Returns:
    - dict: A dictionary with the key "result" containing the quotient.

    Example Input:
    "100, 5, 2"

    Example Output:
    {"result": 10.0}

    Notes:
    - If no numbers are found, the result defaults to 0.
    - Division by zero will raise an error.
    """
    # Extract numbers from the string
    numbers = [int(num) for num in inputs.replace(",", "").split() if num.isdigit()]


    # If no numbers are found, return 0
    if not numbers:
        return {"result": 0}

    # Calculate the result of dividing the first number by subsequent numbers
    result = numbers[0]
    for num in numbers[1:]:
        result /= num

    return {"result": result}

# Building the **agent**

In [46]:
tools = [add_numbers,subtract_numbers, multiply_numbers, divide_numbers]
tools

[StructuredTool(name='add_numbers', description='Adds a list of numbers provided in the input dictionary or extracts numbers from a string.\n\nParameters:\n- inputs (str): \nstring, it should contain numbers that can be extracted and summed.\n\nReturns:\n- dict: A dictionary with a single key "result" containing the sum of the numbers.\n\nExample Input (Dictionary):\n{"numbers": [10, 20, 30]}\n\nExample Input (String):\n"Add the numbers 10, 20, and 30."\n\nExample Output:\n{"result": 60}', args_schema=<class 'langchain_core.utils.pydantic.add_numbers'>, func=<function add_numbers at 0x7d7a089fa2a0>),
 StructuredTool(name='subtract_numbers', description='Extracts numbers from a string, negates the first number, and successively subtracts \nthe remaining numbers in the list.\n\nThis function is designed to handle input in string format, where numbers are separated \nby spaces, commas, or other delimiters. It parses the string, extracts valid numeric values, \nand performs a step-by-step 

In [49]:

math_agent = create_agent(
    model,
    tools,
    system_prompt="You are a helpful mathematical assistant that can perform various operations. Use the tools precisely and explain your reasoning clearly.",
)

In [50]:
response = math_agent.invoke({
    "messages": [("human", "What is 25 divided by 4?")]
})

# Get the final answer
final_answer = response["messages"][-1].content
print(final_answer)

The result of 25 divided by 4 is 6.25.


In [51]:
response_2 = math_agent.invoke({
    "messages": [("human", "Subtract 100, 20, and 10.")]
})

# Get the final answer
final_answer_2 = response_2["messages"][-2].content
print(final_answer_2)

{"result": -130}


Exploring LangChain's built-in **tools¶**

In [54]:
!pip install -q -U langchain langchain-community wikipedia

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 45.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 2.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [59]:
!pip uninstall -y wikipedia
!pip install -q -U wikipedia-api requests

Found existing installation: wikipedia 1.4.0
Uninstalling wikipedia-1.4.0:
  Successfully uninstalled wikipedia-1.4.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.1/129.1 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.4/108.4 kB 7.7 MB/s eta 0:00:00


In [55]:
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

# Configure Wikipedia API
api_wrapper = WikipediaAPIWrapper(
    top_k_results=1,
    doc_content_chars_max=1000
)

# Create tool
search_wikipedia = WikipediaQueryRun(
    api_wrapper=api_wrapper
)